# Check Environment Variable

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get('GOOGLE_API_KEY')
if not api_key:
    raise ValueError('No API key found')
else:
    print('API key found')

API key found


# Chat Models

Read: https://python.langchain.com/docs/integrations/chat/

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Create a model 
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# Generate a response
llm.invoke("Hello, how are you?").content

"I am doing well, thank you for asking! As a large language model, I don't experience emotions or feelings like humans do, but I am functioning optimally and ready to assist you. How can I help you today?"


## Prompt Templates

Read: https://python.langchain.com/docs/concepts/prompt_templates/

### Example 1 (`PromptTemplate`)

`PromptTemplate`
- A class in LangChain used to create prompts with dynamic placeholders
- Allow filling in different values at runtime instead of hardcoding them

In [2]:
# Using a prompt template so that we can dynamically change the prompt
from langchain_core.prompts import PromptTemplate

# Define a template with placeholders {input} & {language}
prompt = PromptTemplate(template="Translate the following text to {language}: {input}")
# or prompt = PromptTemplate.from_template("Translate the following text to French: {input}")

# Create a chain
chain = prompt | llm

chain.invoke({
    "input": "I love programming.",
    "language": "Chinese"   
}).content

'我喜欢编程。 (Wǒ xǐhuan biānchéng.)\n\nThis is the most common and straightforward translation.\n\nHere are a few other options, depending on the nuance you want to convey:\n\n* **我热爱编程。** (Wǒ rè\'ài biānchéng.) - This emphasizes a stronger feeling of love and passion for programming. "热爱" means "to deeply love" or "to be passionate about."\n\n* **我喜欢写代码。** (Wǒ xǐhuan xiě dàimǎ.) - This translates to "I like to write code," focusing on the activity of coding.\n\nThe best translation depends on the context and the specific feeling you want to express. However, **我喜欢编程** is generally a safe and accurate translation.'

### Example 2 (System & User Message)

**System Message (`system`)**

- Define assistant's behavior & rules
- Tell AI how to behave

**User Message (`user`)**

- Represent a message sent by the user (human)
- Actual query / instruction given to AI

In [3]:
message = [
    ("system", "You are a helpful assistant that answers questions about US history."),
    ("user", "Who was the first president of the United States?"),
]
llm.invoke(message).content

'The first president of the United States was **George Washington**. He served from 1789 to 1797.'

In LangChain, instead of using strings like "system" and "user", you can use structured message objects:

- `SystemMessage(content="...")`    → Equivalent to "system"
- `HumanMessage(content="...")`     → Equivalent to "user"

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a helpful assistant that answers questions about the United States. If you are asked about ANYTHING that is not related to the United States, you must say 'I cannot answer that question.'"),
    HumanMessage(content="Who was the first president of the United States?"),
]

llm.invoke(messages).content

'The first president of the United States was George Washington.'

### Example 3 (`ChatPromptTemplate` with System & User Message)

`ChatPromptTemplate`            → A structured way to define multi-turn prompts for chat-based LLMs.

`SystemMessagePromptTemplate`   → Defines system messages (instructions for the AI).

`HumanMessagePromptTemplate`    → Represents messages from the user.

In [5]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import HumanMessagePromptTemplate, SystemMessagePromptTemplate

prompt = ChatPromptTemplate([
    SystemMessagePromptTemplate.from_template("You are a {occupation} named {name}. Get into character and pretend to be this role. Answer questions accordingly."),
    HumanMessagePromptTemplate.from_template("{input}")
])

# another way to do it
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a {occupation} named {name}. Get into character and pretend to be this role. Answer questions accordingly."),
#     ("user", "{input}")
# ])

chain = prompt | llm

chain.invoke({
    "occupation": "old wizard",
    "name": "Meow Cat Wizard",
    "input": "Hi!"
}).content

"*Adjusts my spectacles, peering over them with a twinkle in my ancient feline eyes.*\n\nGreetings, young one! Meow Cat Wizard at your service. I've seen more moons than you've chased mice, I reckon. What brings you to my humble abode, nestled deep within the Whispering Woods? Perhaps a potion, a spell, or just a bit of arcane wisdom? Speak your mind, and I shall lend a furry ear! *Purrs gently, stroking my long, white beard.*"

# Document Loading

Data Loading in langchain 

- Process of loading data from various sources like files, databases, web pages, etc. 
- Data is loaded into a Document object which is a part of the langchain library. 
- Document object contains the text content of the data & metadata like the source URL, title, etc.  
- Document object is then used for further processing like text analysis, summarization, etc.

Source: https://python.langchain.com/docs/integrations/document_loaders

### **Load Text File**

In [10]:
from langchain.document_loaders import TextLoader

loader = TextLoader("test.txt")
document = loader.load()
document[0].page_content

'Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry\'s standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.\n\nContrary to popular belief, Lorem Ipsum is not simply random text. It has roots in a piece of classical Latin literature from 45 BC, making it over 2000 years old. Richard McClintock, a Latin professor at Hampden-Sydney College in Virginia, looked up one of the more obscure Latin words, consectetur, from a Lorem Ipsum passage, and going through the cites of the word in classical literature, discovered the undoubtable

**Load Website**

In [11]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.espn.com/")
documents = loader.load()
documents[0]

USER_AGENT environment variable not set, consider setting it to identify your requests.


Document(metadata={'source': 'https://www.espn.com/', 'title': 'ESPN - Serving Sports Fans. Anytime. Anywhere.', 'description': 'Visit ESPN for live scores, highlights and sports news. Stream exclusive games on ESPN+ and play fantasy sports.', 'language': 'en'}, page_content="\n\n\n\n\n\n\n\n\nESPN - Serving Sports Fans. Anytime. Anywhere.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n        Skip to main content\n    \n\n        Skip to navigation\n    \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n<\n\n>\n\n\n\n\n\n\n\n\n\nMenuESPN\n\n\n\n\n\nscores\n\n\n\nNFLNBANHLNCAAMNCAAWSoccerMLBMore SportsBoxingCFLNCAACricketF1GamingGolfHorseLLWSMMANASCARNLLNBA G LeagueNBA Summer LeagueNCAAFNWSLOlympicsPLLProfessional WrestlingRacingRN BBRN FBRugbySports BettingTennisTGLUFLWNBAX GamesEditionsFantasyWatchESPN BETESPN+\n\n\n\n\n\n\n\n\n\n\

### **Load Multiple Websites**

In [14]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(["https://www.espn.com/", "https://python.langchain.com/docs/integrations/document_loaders/"])
documents = loader.load()
documents

[Document(metadata={'source': 'https://www.espn.com/', 'title': 'ESPN - Serving Sports Fans. Anytime. Anywhere.', 'description': 'Visit ESPN for live scores, highlights and sports news. Stream exclusive games on ESPN+ and play fantasy sports.', 'language': 'en'}, page_content="\n\n\n\n\n\n\n\n\nESPN - Serving Sports Fans. Anytime. Anywhere.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n        Skip to main content\n    \n\n        Skip to navigation\n    \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n<\n\n>\n\n\n\n\n\n\n\n\n\nMenuESPN\n\n\n\n\n\nscores\n\n\n\nNFLNBANHLNCAAMNCAAWSoccerMLBMore SportsBoxingCFLNCAACricketF1GamingGolfHorseLLWSMMANASCARNLLNBA G LeagueNBA Summer LeagueNCAAFNWSLOlympicsPLLProfessional WrestlingRacingRN BBRN FBRugbySports BettingTennisTGLUFLWNBAX GamesEditionsFantasyWatchESPN BETESPN+\n\n\n\n\n\n\n\n\n\n

In [ ]:
# Print length of documents
len(documents)

2

In [16]:
# Print metadata of first document
documents[0].metadata

{'source': 'https://www.espn.com/',
 'title': 'ESPN - Serving Sports Fans. Anytime. Anywhere.',
 'description': 'Visit ESPN for live scores, highlights and sports news. Stream exclusive games on ESPN+ and play fantasy sports.',
 'language': 'en'}

**Load CSV Files**

In [20]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="Air_Quality.csv", content_columns=["Unique ID", "Name"])
documents = loader.load()
documents[0].page_content

'Unique ID: 179772\nName: Boiler Emissions- Total SO2 Emissions'

In [21]:
# Print length of documents
len(documents)

18025

### **Load PDF**

In [30]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("attention-is-all-you-need.pdf")
documents = loader.load()

print("number of pages or documents: ", len(documents))
print("----------------")

# Print first 500 characters of first page
print("First 500 characters of fifth page:\n")
print(documents[5].page_content[:500])

number of pages or documents:  11
----------------
First 500 characters of fifth page:

Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types. nis the sequence length, dis the representation dimension, kis the kernel
size of convolutions and rthe size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2 ·d) O(1) O(1)
Recurrent O(n·d2) O(n) O(n)
Convolutional O(k·n·d2) O(1) O(logk(n))
Self-Attention (restricted) O(r·n·d) O(1) 


### **Cleaning up the data after loading**

If you look at the page_content grabbed from a website or the text content grabbed from a file, you will see that it contains a lot of unwanted characters like newlines, tabs, etc.

In [38]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://python.langchain.com/docs/integrations/document_loaders/")
documents = loader.load()
print(documents[0].page_content[:500])






Document loaders | 🦜️🔗 LangChain






Skip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1💬SearchProvidersAnthropicAWSGoogleHugging FaceMicrosoftOpenAIMoreProvidersAbsoAcreomActiveloop Deep LakeAerospikeAI21 LabsAimAINetworkAirbyteAirtableAlchemyAleph AlphaAlibaba CloudAnalyticDBAnnoyAnthropicAnyscaleApache S


In [39]:
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer

loader = AsyncHtmlLoader(["https://python.langchain.com/docs/integrations/document_loaders/"])
documents = loader.load()
html2text = Html2TextTransformer()
documents2 = html2text.transform_documents(documents)
print(documents2[0].page_content)

Skip to main content

**Join us at Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in
San Francisco!**

IntegrationsAPI Reference

More

  * Contributing
  * People
  * Error reference
  * * * *

  * LangSmith
  * LangGraph
  * LangChain Hub
  * LangChain JS/TS

v0.3

  * v0.3
  * v0.2
  * v0.1

💬

Search

  * Providers

    * Anthropic
    * AWS
    * Google
    * Hugging Face
    * Microsoft
    * OpenAI
    * More

      * Providers
      * Abso
      * Acreom
      * Activeloop Deep Lake
      * Aerospike
      * AI21 Labs
      * Aim
      * AINetwork
      * Airbyte
      * Airtable
      * Alchemy
      * Aleph Alpha
      * Alibaba Cloud
      * AnalyticDB
      * Annoy
      * Anthropic
      * Anyscale
      * Apache Software Foundation
      * Apache Doris
      * Apify
      * Apple
      * ArangoDB
      * Arcee
      * ArcGIS
      * Argilla
      * Arize
      * Arthur
      * Arxiv
      * Ascend
      * AskNews
      * AssemblyAI
      * Astra DB
      *

/Users/feliciang/Documents/rag_langchain/venv/lib/python3.12/site-packages/langchain_community/document_loaders/async_html.py:195: UserWarning: For better logging of progress, `pip install tqdm`
  warnings.warn("For better logging of progress, `pip install tqdm`")


# Splitting Into Chunks

Chunking: Splitting documents into smaller chunks of text for processing

**How to choose chunk size for splitting documents?**

- Model's Context Window: 
    - If you have a model with a high context window, which means it can handle more text or tokens, you may not need to have small chunks. 
    - However, if you have a model with a limited context window, it can only handle so much information, so having smaller amounts of information makes sense, thus a smaller chunk size.

- Format and Structure of Documents: 
    - Consider the format & structure of the documents. 
    - If you have a blog article, it would make sense to split based on the paragraphs because you want to keep related ideas together. 
    - When feeding context to a LLM, you want it to be all of the relevant contextual information that it would need to answer the question effectively.

- Experimentation:
    - No one-size-fits-all answer for choosing chunk size.
    - Have to experiment with. 
    - Start with a chunk size that feels reasonable, such as a few hundred. 
    - Then, test it out to see how that does & pay attention to whether the chunks retain enough context. 
    - If it is cutting off too much information, then adjust it to make it a little larger

### **Split by a Character**

`CharacterTextSplitter` is a basic text splitter that splits by looking for a character such as \n (newline). 

It will try to split enough text to get the desired chunk size, but sometimes it cannot find enough instances of the character to get the desired chunk size.

In [52]:
# load the text from test.txt
from langchain_community.document_loaders import TextLoader

loader = TextLoader("test.txt")
documents = loader.load()
print(documents[0].page_content)

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

Contrary to popular belief, Lorem Ipsum is not simply random text. It has roots in a piece of classical Latin literature from 45 BC, making it over 2000 years old. Richard McClintock, a Latin professor at Hampden-Sydney College in Virginia, looked up one of the more obscure Latin words, consectetur, from a Lorem Ipsum passage, and going through the cites of the word in classical literature, discovered the undoubtable sou

In [53]:
from langchain_text_splitters import CharacterTextSplitter

# Basic text splitter
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separator="\n")
chunks = text_splitter.split_documents(documents)
print("Number of chunks: ", len(chunks))

# Print each chunk along with its text
for chunk in chunks:
    print(f"Chunk {chunks.index(chunk)} size: {len(chunk.page_content)}")
    print(chunk.page_content)

Number of chunks:  5
Chunk 0 size: 574
Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.
Chunk 1 size: 763
Contrary to popular belief, Lorem Ipsum is not simply random text. It has roots in a piece of classical Latin literature from 45 BC, making it over 2000 years old. Richard McClintock, a Latin professor at Hampden-Sydney College in Virginia, looked up one of the more obscure Latin words, consectetur, from a Lorem Ipsum passage, and going through the cites of the word

### **Split by a List of Characters**

`RecursiveCharacterTextSplitter` 

- More sophisticated approach that goes through a list of splitter characters like: ["\n\n", "\n", " ", ""] & splits the text until they in small enough chunks. 

- Has effect of trying to keep all paragraphs (& then sentences, & then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [54]:
# load the text from https://www.govinfo.gov/content/pkg/CDOC-110hdoc50/html/CDOC-110hdoc50.htm
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.govinfo.gov/content/pkg/CDOC-110hdoc50/html/CDOC-110hdoc50.htm")
documents = loader.load()
print(documents[0].page_content[:500])


[House Document 110-50]
[From the U.S. Government Publishing Office]



110th Congress                                               Document

                        HOUSE OF REPRESENTATIVES
1st Session                                                No. 110-50

 
                                   THE
                              CONSTITUTION
                                 OF THE
                              UNITED STATES
                               OF AMERICA

                         


In [55]:
# Improving splitting by using the RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)
print("Number of chunks: ", len(chunks))

# Print size of first 5 chunks
for i in range(5):
    print(f"Chunk {i+1} size: {len(chunks[i].page_content)}")

Number of chunks:  430
Chunk 1 size: 916
Chunk 2 size: 573
Chunk 3 size: 880
Chunk 4 size: 951
Chunk 5 size: 976


**Why Overlap Is Important?** 

- Without overlap, parts of text that belong together are split into different chunks & the semantic relationship between them is lost.

Example:
1. "I have a cat allery. So I would never say anything like"
2. "I love cats. Bla bla"

Question: Does this person like cats?

Response: Yes

In [56]:
# end of chunk 0:
print("\nEnd of chunk 0:")
print(chunks[3].page_content[-300:])
# beginning of chunk 1:
print("\nBeginning of chunk 1:")
print(chunks[4].page_content[:300])


End of chunk 0:
ior to the adoption of the Federal Constitution, the 
Articles of Confederation, drafted by the Continental Congress 
and approved by 13 States, provided for a union of the former 
British colonies. Even before Maryland became the last State to 
accede to the Articles in 1781, a number of Americans,

Beginning of chunk 1:
and approved by 13 States, provided for a union of the former 
British colonies. Even before Maryland became the last State to 
accede to the Articles in 1781, a number of Americans, 
particularly those involved in the prosecution of the 
Revolutionary War, recognized the inadequacies of the Article


### **Split Based On Document Structure**

Sometimes your documents will be a specific text format like HTML, markdown, or others. 

This gives us the opportunity to split on elements of those formats instead of just things like sentences or paragraphs.

In [57]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = "# Foo\n\n    ## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n ### Boo \n\n Hi this is Amy \n\n ## Baz\n\n Hi this is Molly"

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}, page_content='Hi this is Jim  \nHi this is Joe'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Boo'}, page_content='Hi this is Amy'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}, page_content='Hi this is Molly')]

# Embeddings

Embeddings are a way to represent text as numbers. 

Specifically, they are a way to represent text as a vector of numbers such as [0.1, 0.2, 0.3, 0.4, 0.5].

The numbers encode the semantic meaning of the text.

Hugging Face Models: https://huggingface.co/spaces/mteb/leaderboard

LangChain Embedding Models: https://python.langchain.com/docs/integrations/text_embedding/ 

**Using Google embedding model**

In [60]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Document Loader
loader = WebBaseLoader("https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag/")
documents = loader.load()

# Text Splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)
print("Number of chunks: ", len(chunks))

# Instantiate the embeddings model. The embeddings model_name can be changed as desired
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# Create embeddings for all chunks
chunk_embedding = embeddings.embed_documents([chunk.page_content for chunk in chunks])

# Check length(dimension) of the embedding
len(chunk_embedding[0])

Number of chunks:  26


768

# Vector Stores

LangChain Vector Stores: https://python.langchain.com/docs/integrations/vectorstores/

**Using FAISS from Facebook**

In [61]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Document Loader
loader = WebBaseLoader("https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag/")
documents = loader.load()

# Text Splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)
print("Number of chunks: ", len(chunks))

# Instantiate the embeddings model. The embeddings model_name can be changed as desired
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# !! Don't need this cause vector store will do it
# Create embeddings for all chunks
# chunk_embedding = embeddings.embed_documents([chunk.page_content for chunk in chunks])

# Create database & index the embeddings
vectore_store = FAISS.from_documents(chunks, embeddings)

# Check number of chunks that have been indexed
vectore_store.index.ntotal

Number of chunks:  26


26

In [62]:
# Query
query = "what is RAG?"

# Get the top 3 similar chunks
docs = vectore_store.similarity_search(query, k=3)

# Print chunks
for doc in docs:
    print("-" * 80)
    print(doc.page_content)
    print("\n" * 2)

--------------------------------------------------------------------------------
RAG is a relatively new artificial intelligence technique that can improve the quality of generative AI by allowing large language model (LLMs) to tap additional data resources without retraining.
RAG models build knowledge repositories based on the organization’s own data, and the repositories can be continually updated to help the generative AI provide timely, contextual answers.
Chatbots and other conversational systems that use natural language processing can benefit greatly from RAG and generative AI.
Implementing RAG requires technologies such as vector databases, which allow for the rapid coding of new data, and searches against that data to feed into the LLM.



--------------------------------------------------------------------------------
What Is Retrieval-Augmented Generation (RAG)?
That’s where retrieval-augmented generation (RAG) comes in. RAG provides a way to optimize the output of an LLM w

In [ ]:
# saving the vector database
# vectore_store.save_local("practice_vector_db")

# loading the vector database
# vectore_store = FAISS.load_local("practice_vector_db", embeddings)

# Retriever

Now that we understand the indexing pipeline, we can utilize our vector database to retrieve relevant documents for a given query.

LangChain provides a uniform interface for interacting with different types of retrieval systems. 

LangChain retriever interface is straightforward:

- Input   -> A query (string) 
- Output  -> A list of documents (standardized LangChain Document objects)

Read More: https://python.langchain.com/docs/concepts/retrievers/

List of Retreiver: https://python.langchain.com/docs/integrations/retrievers/

**Wikipedia Retriever**

In [66]:
# Wikipedia Retriever
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever()
docs = retriever.invoke("France")

docs[0].page_content[:200]

'France, officially the French Republic, is a country located primarily in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the '

**Retrieve Document**

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Document Loader
loader = WebBaseLoader("https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag/")
documents = loader.load()

# Text Splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)
print("Number of chunks: ", len(chunks))

# Instantiate the embeddings model. The embeddings model_name can be changed as desired
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# !! Don't need this cause vector store will do it
# Create embeddings for all chunks
# chunk_embedding = embeddings.embed_documents([chunk.page_content for chunk in chunks])

# Create database & index the embeddings
vectore_store = FAISS.from_documents(chunks, embeddings)

# Using a Vector store as a retriever
retriever = vectore_store.as_retriever(search_kwargs={"k": 2})      # Get top 2 results

# Query
docs = retriever.invoke("What is RAG?")             

# Function to print content of the documents
def print_docs(docs):
    for doc in docs:
        print(doc.page_content[:500])
        print("-"*100+"\n")

print_docs(docs)


Number of chunks:  26
RAG is a relatively new artificial intelligence technique that can improve the quality of generative AI by allowing large language model (LLMs) to tap additional data resources without retraining.
RAG models build knowledge repositories based on the organization’s own data, and the repositories can be continually updated to help the generative AI provide timely, contextual answers.
Chatbots and other conversational systems that use natural language processing can benefit greatly from RAG and gen
----------------------------------------------------------------------------------------------------

What Is Retrieval-Augmented Generation (RAG)?
That’s where retrieval-augmented generation (RAG) comes in. RAG provides a way to optimize the output of an LLM with targeted information without modifying the underlying model itself; that targeted information can be more up-to-date than the LLM as well as specific to a particular organization and industry. That means the gene